# Neo4j Azure Connectivity Test

Validates connectivity from this Databricks workspace to a Neo4j cluster deployed with Azure VNet peering.
Credentials are read from Databricks Secrets — no hardcoded values.

**Tests:**
1. Network (TCP socket to Bolt port) — confirms VNet peering and NSG rules
2. Python driver (authenticate + execute query) — confirms Bolt protocol end-to-end
3. Cluster topology (`SHOW SERVERS`) — confirms all cluster nodes are online

In [ ]:
%pip install neo4j --quiet

---
## Configuration

Connection details are loaded from the Databricks Secrets scope created by:
```
neo4j-deploy-ansible setup-databricks --scenario <scenario> --token <pat>
```

In [ ]:
SCOPE_NAME = "SCOPE_NAME_PLACEHOLDER"

BOLT_URI  = dbutils.secrets.get(SCOPE_NAME, "bolt_uri")
HOST      = dbutils.secrets.get(SCOPE_NAME, "host")
USERNAME  = dbutils.secrets.get(SCOPE_NAME, "username")
PASSWORD  = dbutils.secrets.get(SCOPE_NAME, "password")
DATABASE  = dbutils.secrets.get(SCOPE_NAME, "database")
BOLT_PORT = 7687

print(f"Config loaded from scope : {SCOPE_NAME}")
print(f"  Bolt URI : {BOLT_URI}")
print(f"  Host     : {HOST}")
print(f"  Database : {DATABASE}")
print(f"  Username : {USERNAME}")

---
## Section 1: Network Connectivity (TCP)

**Expected result:** PASS — confirms VNet peering is active and NSG allows Bolt traffic on port 7687.

In [ ]:
import socket
import time

print("=" * 60)
print("TEST: Network Connectivity (TCP)")
print("=" * 60)
print(f"\nTarget: {HOST}:{BOLT_PORT}")

try:
    start = time.time()
    sock = socket.create_connection((HOST, BOLT_PORT), timeout=10)
    sock.close()
    elapsed = (time.time() - start) * 1000

    print("\n" + "=" * 60)
    print(">>> CONNECTIVITY VERIFIED <<<")
    print("=" * 60)
    print(f"[PASS] TCP connection established in {elapsed:.1f}ms")
    print(f"\nDetails:")
    print(f"  Host : {HOST}")
    print(f"  Port : {BOLT_PORT}")
    print(f"  RTT  : {elapsed:.1f}ms")
    print("\nRESULT: VNet peering + NSG rules are OPEN")
    print("Status: PASS")

except Exception as e:
    print(f"\n[FAIL] Cannot reach {HOST}:{BOLT_PORT}")
    print(f"Error : {e}")
    print("\nCheck:")
    print("  - VNet peering state (both directions must be Connected)")
    print("  - NSG inbound rule for port 7687 from Databricks CIDR")
    print("  - Neo4j VMSS instances are running")
    print("Status: FAIL")

---
## Section 2: Neo4j Python Driver

**Expected result:** PASS — confirms Bolt authentication and query execution over the peered network.

In [ ]:
import time
from neo4j import GraphDatabase

print("=" * 60)
print("TEST: Neo4j Python Driver")
print("=" * 60)
print(f"\nTarget: {BOLT_URI}")

try:
    start = time.time()
    driver = GraphDatabase.driver(BOLT_URI, auth=(USERNAME, PASSWORD))
    driver.verify_connectivity()
    connect_time = (time.time() - start) * 1000

    print("\n" + "=" * 60)
    print(">>> AUTHENTICATION SUCCESSFUL <<<")
    print("=" * 60)
    print(f"[PASS] Driver connected in {connect_time:.1f}ms")

    with driver.session(database=DATABASE) as session:
        q_start = time.time()
        rec = session.run("RETURN 1 AS test").single()
        q_time = (time.time() - q_start) * 1000
        print(f"[PASS] RETURN 1 = {rec['test']} ({q_time:.1f}ms)")

        components = session.run(
            "CALL dbms.components() YIELD name, versions RETURN name, versions"
        )
        for rec in components:
            print(f"[INFO] {rec['name']} {rec['versions']}")

    driver.close()
    total = (time.time() - start) * 1000
    print(f"\nTotal test time: {total:.1f}ms")
    print("RESULT: Python driver WORKING")
    print("Status: PASS")

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("Status: FAIL")

---
## Section 3: Cluster Topology

**Expected result:** 3 ENABLED servers — confirms all cluster nodes are online and visible via the load balancer.

In [ ]:
from neo4j import GraphDatabase

print("=" * 60)
print("TEST: Cluster Topology (SHOW SERVERS)")
print("=" * 60)

try:
    driver = GraphDatabase.driver(BOLT_URI, auth=(USERNAME, PASSWORD))

    with driver.session(database="system") as session:
        result = session.run("SHOW SERVERS")
        servers = result.data()

    driver.close()

    enabled = [s for s in servers if s.get("state", "").upper() == "ENABLED"]

    print(f"\nFound {len(servers)} server(s), {len(enabled)} enabled:\n")
    for s in servers:
        state  = s.get("state", "unknown")
        name   = s.get("name", s.get("serverId", "unknown"))
        addr   = s.get("address", "unknown")
        marker = "✓" if state.upper() == "ENABLED" else "✗"
        print(f"  {marker} {name:<36} {state:<10} {addr}")

    print()
    if len(enabled) >= 3:
        print(f"[PASS] All {len(enabled)} nodes ENABLED")
        print("RESULT: Cluster healthy")
        print("Status: PASS")
    elif len(enabled) > 0:
        print(f"[WARN] Only {len(enabled)}/{len(servers)} nodes enabled — cluster may be degraded")
        print("Status: WARN")
    else:
        print("[FAIL] No enabled nodes found")
        print("Status: FAIL")

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("Status: FAIL")